# Part B: Dataset Understanding & Preparation

## Q6: Identify Features & Target Variables

Features (X):  
age, country_region, device_type, education_background, course_level, course_category, course_start_date, week_of_year, sessions, time_spent_hours, videos_watched, quiz_attempts, assignments_submitted, forum_posts, avg_quiz_score, attendance_rate

Target Variables (y):

Classification Task: completion_status (0 = Not Completed, 1 = Completed)

Regression Task: final_score (continuous 0–100)

## Q7: Train–Test Split

In [8]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv("Smart_Outcome_Predictor_Dataset_5200(Sheet1).csv")

X = df.drop(["completion_status", "final_score"], axis=1)
y_class = df["completion_status"]
y_reg   = df["final_score"]

# Classification split
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_class, test_size=0.2, random_state=42)

# Regression split
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42)


## Q8: Basic Preprocessing

We handle categorical + numerical features separately.

Categorical: country_region, device_type, education_background, course_level, course_category

Numerical: age, week_of_year, sessions, time_spent_hours, videos_watched, quiz_attempts, assignments_submitted, forum_posts, avg_quiz_score, attendance_rate

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_cols = ["country_region", "device_type", "education_background", "course_level", "course_category"]
num_cols = ["age", "week_of_year", "sessions", "time_spent_hours", "videos_watched",
            "quiz_attempts", "assignments_submitted", "forum_posts", "avg_quiz_score", "attendance_rate"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ])


# Part C: Bagging (Bootstrap Aggregating)

## Q9: Implement Bagging Classifier (Course Completion Prediction)

In [11]:

from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

# Remove rows where completion_status is NaN
df_class = df.dropna(subset=["completion_status"])

X_class = df_class.drop(["completion_status", "final_score"], axis=1)
y_class = df_class["completion_status"]

# Train-test split
from sklearn.model_selection import train_test_split
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class, y_class, test_size=0.2, random_state=42
)

# Bagging Classifier pipeline
bag_clf = Pipeline([
    ("preprocessor", preprocessor),   # use your preprocessing pipeline
    ("model", BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        n_estimators=50,
        random_state=42))
])

# Train
bag_clf.fit(X_train_c, y_train_c)

# Predict
y_pred_c = bag_clf.predict(X_test_c)

# Evaluate
print("Bagging Classifier Accuracy:", accuracy_score(y_test_c, y_pred_c))
print("Confusion Matrix:\n", confusion_matrix(y_test_c, y_pred_c))


Bagging Classifier Accuracy: 0.7076923076923077
Confusion Matrix:
 [[537 110]
 [194 199]]


## Q10: Implement Bagging Regressor (Final Score Prediction)

In [12]:
# Remove rows where final_score is NaN
df_reg = df.dropna(subset=["final_score"])

X_reg = df_reg.drop(["completion_status", "final_score"], axis=1)
y_reg = df_reg["final_score"]

# Train-test split
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42)

from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Define categorical & numerical columns
cat_cols = ["country_region", "device_type", "education_background", "course_level", "course_category"]
num_cols = ["age", "week_of_year", "sessions", "time_spent_hours", "videos_watched",
            "quiz_attempts", "assignments_submitted", "forum_posts", "avg_quiz_score", "attendance_rate"]

# Preprocessor with imputation
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler())
        ]), num_cols),
        
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ])

# Bagging Regressor pipeline
bag_reg = Pipeline([
    ("preprocessor", preprocessor),
    ("model", BaggingRegressor(
        estimator=DecisionTreeRegressor(),
        n_estimators=50, random_state=42))
])

# Train
bag_reg.fit(X_train_r, y_train_r)

# Predict
y_pred_r = bag_reg.predict(X_test_r)

# Evaluate
print("Bagging Regressor MSE:", mean_squared_error(y_test_r, y_pred_r))
print("Bagging Regressor R²:", r2_score(y_test_r, y_pred_r))


Bagging Regressor MSE: 101.26154526538463
Bagging Regressor R²: 0.45802942733753615


## Q11: Compare Bagging Results with Single Base Model

In [13]:
# Single Decision Tree Classifier
dt_clf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42))
])
dt_clf.fit(X_train_c, y_train_c)
y_pred_dt_c = dt_clf.predict(X_test_c)
print("Decision Tree Accuracy:", accuracy_score(y_test_c, y_pred_dt_c))

# Single Decision Tree Regressor
dt_reg = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(random_state=42))
])
dt_reg.fit(X_train_r, y_train_r)
y_pred_dt_r = dt_reg.predict(X_test_r)
print("Decision Tree MSE:", mean_squared_error(y_test_r, y_pred_dt_r))
print("Decision Tree R²:", r2_score(y_test_r, y_pred_dt_r))


Decision Tree Accuracy: 0.6326923076923077
Decision Tree MSE: 202.240375
Decision Tree R²: -0.0824279993650272


# Part D: Boosting Algorithms

## Q12: AdaBoost Classifier (Course Completion Prediction)

In [14]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

# AdaBoost Classifier pipeline
ada_clf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=100, learning_rate=0.8, random_state=42))
])

# Train
ada_clf.fit(X_train_c, y_train_c)

# Predict
y_pred_c = ada_clf.predict(X_test_c)

# Evaluate
print("AdaBoost Classifier Accuracy:", accuracy_score(y_test_c, y_pred_c))
print("Confusion Matrix:\n", confusion_matrix(y_test_c, y_pred_c))


AdaBoost Classifier Accuracy: 0.7336538461538461
Confusion Matrix:
 [[543 104]
 [173 220]]


## Q13: AdaBoost Regressor (Final Score Prediction)

In [15]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

# AdaBoost Regressor pipeline
ada_reg = Pipeline([
    ("preprocessor", preprocessor),
    ("model", AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=3),
        n_estimators=100, learning_rate=0.8, random_state=42))
])

# Train
ada_reg.fit(X_train_r, y_train_r)

# Predict
y_pred_r = ada_reg.predict(X_test_r)

# Evaluate
print("AdaBoost Regressor MSE:", mean_squared_error(y_test_r, y_pred_r))
print("AdaBoost Regressor R²:", r2_score(y_test_r, y_pred_r))


AdaBoost Regressor MSE: 113.05968393752245
AdaBoost Regressor R²: 0.3948836007976413


## Q14: Gradient Boosting Classifier

In [16]:
from sklearn.ensemble import GradientBoostingClassifier

gb_clf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42))
])

gb_clf.fit(X_train_c, y_train_c)
y_pred_c = gb_clf.predict(X_test_c)

print("Gradient Boosting Classifier Accuracy:", accuracy_score(y_test_c, y_pred_c))
print("Confusion Matrix:\n", confusion_matrix(y_test_c, y_pred_c))


Gradient Boosting Classifier Accuracy: 0.7153846153846154
Confusion Matrix:
 [[521 126]
 [170 223]]


## Q15: Gradient Boosting Regressor

In [17]:
from sklearn.ensemble import GradientBoostingRegressor

gb_reg = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42))
])

gb_reg.fit(X_train_r, y_train_r)
y_pred_r = gb_reg.predict(X_test_r)

print("Gradient Boosting Regressor MSE:", mean_squared_error(y_test_r, y_pred_r))
print("Gradient Boosting Regressor R²:", r2_score(y_test_r, y_pred_r))


Gradient Boosting Regressor MSE: 98.71318393730631
Gradient Boosting Regressor R²: 0.47166872984580643


## Q16: LightGBM Classifier

In [18]:
from lightgbm import LGBMClassifier

lgb_clf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=-1, random_state=42))
])

lgb_clf.fit(X_train_c, y_train_c)
y_pred_c = lgb_clf.predict(X_test_c)

print("LightGBM Classifier Accuracy:", accuracy_score(y_test_c, y_pred_c))
print("Confusion Matrix:\n", confusion_matrix(y_test_c, y_pred_c))


[LightGBM] [Info] Number of positive: 1559, number of negative: 2601
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000245 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1074
[LightGBM] [Info] Number of data points in the train set: 4160, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.374760 -> initscore=-0.511851
[LightGBM] [Info] Start training from score -0.511851
LightGBM Classifier Accuracy: 0.7019230769230769
Confusion Matrix:
 [[515 132]
 [178 215]]


## Q17: LightGBM Regressor

In [19]:
from lightgbm import LGBMRegressor

lgb_reg = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMRegressor(
        n_estimators=200, learning_rate=0.1, max_depth=-1, random_state=42))
])

lgb_reg.fit(X_train_r, y_train_r)
y_pred_r = lgb_reg.predict(X_test_r)

print("LightGBM Regressor MSE:", mean_squared_error(y_test_r, y_pred_r))
print("LightGBM Regressor R²:", r2_score(y_test_r, y_pred_r))


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000141 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1074
[LightGBM] [Info] Number of data points in the train set: 4160, number of used features: 30
[LightGBM] [Info] Start training from score 74.774183
LightGBM Regressor MSE: 102.26668915638844
LightGBM Regressor R²: 0.45264971079471805


## Q18: XGBoost Classifier

In [24]:
# Remove rows where target (completion_status) is NaN
df_class = df.dropna(subset=["completion_status"])

# Define X and y again
X = df_class.drop(["completion_status", "final_score"], axis=1)
y = df_class["completion_status"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [25]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

xgb_clf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42,
        use_label_encoder=False,
        eval_metric="logloss"
    ))
])

# Train
xgb_clf.fit(X_train, y_train)

# Predict
y_pred_c = xgb_clf.predict(X_test)

# Evaluate
print("XGBoost Classifier Accuracy:", accuracy_score(y_test, y_pred_c))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_c))


XGBoost Classifier Accuracy: 0.7403846153846154
Confusion Matrix:
 [[543 107]
 [163 227]]


## Q19: XGBoost Regressor

In [26]:
from xgboost import XGBRegressor

xgb_reg = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42))
])

xgb_reg.fit(X_train_r, y_train_r)
y_pred_r = xgb_reg.predict(X_test_r)

print("XGBoost Regressor MSE:", mean_squared_error(y_test_r, y_pred_r))
print("XGBoost Regressor R²:", r2_score(y_test_r, y_pred_r))


XGBoost Regressor MSE: 97.9012102922415
XGBoost Regressor R²: 0.47601456340235826


# Part E: Voting & Stacking Ensembles

## Q24: Voting Classifier (Multiple Base Models)

In [27]:
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
import warnings
warnings.filterwarnings("ignore")

# Define base models
clf1 = LogisticRegression(max_iter=1000, random_state=42)
clf2 = DecisionTreeClassifier(max_depth=5, random_state=42)
clf3 = SVC(probability=True, random_state=42)

# Voting Classifier (default = hard voting)
voting_clf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", VotingClassifier(
        estimators=[("lr", clf1), ("dt", clf2), ("svc", clf3)],
        voting="hard"))
])

voting_clf.fit(X_train_c, y_train_c)
y_pred_c = voting_clf.predict(X_test_c)

print("Voting Classifier Accuracy:", accuracy_score(y_test_c, y_pred_c))


Voting Classifier Accuracy: 0.7278846153846154


## Q25: Hard Voting vs Soft Voting

In [28]:
# Hard Voting
voting_hard = VotingClassifier(
    estimators=[("lr", clf1), ("dt", clf2), ("svc", clf3)],
    voting="hard")

# Soft Voting
voting_soft = VotingClassifier(
    estimators=[("lr", clf1), ("dt", clf2), ("svc", clf3)],
    voting="soft")

pipe_hard = Pipeline([("preprocessor", preprocessor), ("model", voting_hard)])
pipe_soft = Pipeline([("preprocessor", preprocessor), ("model", voting_soft)])

pipe_hard.fit(X_train_c, y_train_c)
pipe_soft.fit(X_train_c, y_train_c)

print("Hard Voting Accuracy:", accuracy_score(y_test_c, pipe_hard.predict(X_test_c)))
print("Soft Voting Accuracy:", accuracy_score(y_test_c, pipe_soft.predict(X_test_c)))


Hard Voting Accuracy: 0.7278846153846154
Soft Voting Accuracy: 0.7192307692307692


## Q26: Stacking Classifier (Meta‑Learner)

In [29]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings("ignore")


stack_clf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", StackingClassifier(
        estimators=[("lr", clf1), ("dt", clf2), ("svc", clf3)],
        final_estimator=LogisticRegression(max_iter=1000),
        passthrough=True))
])

stack_clf.fit(X_train_c, y_train_c)
y_pred_c = stack_clf.predict(X_test_c)

print("Stacking Classifier Accuracy:", accuracy_score(y_test_c, y_pred_c))


Stacking Classifier Accuracy: 0.7259615384615384


## Q27: Stacking Regressor (Score Prediction)

In [30]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR

# Base regressors
reg1 = LinearRegression()
reg2 = DecisionTreeRegressor(max_depth=5, random_state=42)
reg3 = SVR()

stack_reg = Pipeline([
    ("preprocessor", preprocessor),
    ("model", StackingRegressor(
        estimators=[("lr", reg1), ("dt", reg2), ("svr", reg3)],
        final_estimator=LinearRegression()))
])

stack_reg.fit(X_train_r, y_train_r)
y_pred_r = stack_reg.predict(X_test_r)

print("Stacking Regressor MSE:", mean_squared_error(y_test_r, y_pred_r))
print("Stacking Regressor R²:", r2_score(y_test_r, y_pred_r))


Stacking Regressor MSE: 93.91007159784343
Stacking Regressor R²: 0.4973758779873695


# Part F: Model Evaluation & Comparison

## Q28: Classification Model Evaluation

In [31]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

# Example: using Soft Voting Classifier (supports predict_proba)
y_pred_c = pipe_soft.predict(X_test_c)
y_prob_c = pipe_soft.predict_proba(X_test_c)[:,1]  # probability for ROC-AUC

print("Accuracy:", accuracy_score(y_test_c, y_pred_c))
print("Precision:", precision_score(y_test_c, y_pred_c, average="weighted"))
print("Recall:", recall_score(y_test_c, y_pred_c, average="weighted"))
print("F1-Score:", f1_score(y_test_c, y_pred_c, average="weighted"))
print("ROC-AUC:", roc_auc_score(y_test_c, y_prob_c))

print("\nClassification Report:\n", classification_report(y_test_c, y_pred_c))


Accuracy: 0.7192307692307692
Precision: 0.712811815050621
Recall: 0.7192307692307692
F1-Score: 0.7102066435657227
ROC-AUC: 0.7709019117398366

Classification Report:
               precision    recall  f1-score   support

         0.0       0.74      0.84      0.79       647
         1.0       0.67      0.51      0.58       393

    accuracy                           0.72      1040
   macro avg       0.70      0.68      0.68      1040
weighted avg       0.71      0.72      0.71      1040



## Q29: Regression Model Evaluation

In [32]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Example: Stacking Regressor evaluation
y_pred_r = stack_reg.predict(X_test_r)

mae = mean_absolute_error(y_test_r, y_pred_r)
rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_r))
r2 = r2_score(y_test_r, y_pred_r)

print("MAE:", mae)
print("RMSE:", rmse)
print("R² Score:", r2)


MAE: 7.764461097349008
RMSE: 9.690720901865012
R² Score: 0.4973758779873695


## Q30: Compare All Ensemble Techniques

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
| --- | --- | --- | --- | --- | --- |
| Bagging Classifier | … | … | … | … | … |
| AdaBoost Classifier | … | … | … | … | … |
| Gradient Boosting Clf | … | … | … | … | … |
| LightGBM Classifier | … | … | … | … | … |
| XGBoost Classifier | … | … | … | … | … |
| Voting Classifier | … | … | … | … | … |
| Stacking Classifier | … | … | … | … | … |

## Regression Models Comparison

| Model | MAE | RMSE | R² |
| --- | --- | --- | --- |
| Bagging Regressor | … | … | … |
| AdaBoost Regressor | … | … | … |
| Gradient Boosting Reg | … | … | … |
| LightGBM Regressor | … | … | … |
| XGBoost Regressor | … | … | … |
| Stacking Regressor | … | … | … |